#### **Install Dependencies**

In [ ]:
! pip install -q llama-index-core llama-index-embeddings-huggingface llama-index-postprocessor-sbert-rerank llama-index-llms-openai sentence-transformers datasets

#### **Load Dataset**

In [ ]:
from datasets import load_dataset 

# load Amharic Question Answering dataset
amharic_qa = load_dataset("israel/AmharicQA", split="validation").shuffle(seed=42)
amharic_qa

In [ ]:
queries = amharic_qa['question']
documents = amharic_qa['context']
answers = amharic_qa['answers']

len(queries), len(documents), len(answers)

#### **Two Stage Retrieval with Llama Index**

In [ ]:
from llama_index.core import Document, VectorStoreIndex, Settings, QueryBundle
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank
from llama_index.core.schema import TextNode

In [ ]:
# explicitly disable the LLM
Settings.llm = None

nodes = [
    TextNode(text=document, metadata={})
    for document in list(set(documents))
]

print('Number of Nodes:', len(nodes))

In [ ]:
# 1. Configure Embedding
Settings.embed_model = HuggingFaceEmbedding(
    model_name="rasyosef/embedding-amharic-base"
)

# 2. Build Index
index = VectorStoreIndex(nodes, show_progress=True)

In [ ]:
# 3. Setup Retriever and Reranker independently
retriever = index.as_retriever(similarity_top_k=5)
reranker = SentenceTransformerRerank(
    model="rasyosef/reranker-amharic-base",
    top_n=3,
    cross_encoder_kwargs={"max_length": 510} # Forces the tokenizer to clip the input
  )

In [ ]:
idx = 6
query = queries[idx]
print(query)

In [ ]:
# 4. Retrieve initial candidates (Vector Search)
retrieved_nodes = retriever.retrieve(query)

print(f"Query: {query}\n")
print("--- Retrieved Source Nodes ---")
for node in retrieved_nodes:
    print(f"Score: {node.score:.3f} | Text: {node.text}...")

In [ ]:
# 5. Rerank the candidates (Cross-Encoder)
reranked_nodes = reranker.postprocess_nodes(
    retrieved_nodes,
    query_bundle=QueryBundle(query)
)

print(f"Query: {query}\n")
print("--- Reranked Source Nodes ---")
for node in reranked_nodes:
    print(f"Score: {node.score:.3f} | Text: {node.text}")

### **RAG**

In [ ]:
import os
from google.colab import userdata

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from llama_index.llms.openai import OpenAI

# 1. Initialize your LLM
# Ensure your environment is configured for the OpenAI API
llm = OpenAI(model="gpt-5.4-mini")

# 2. Configure the Query Engine
# Passing these arguments constructs a pipeline that performs
# Retrieval -> Reranking -> Synthesis
query_engine = index.as_query_engine(
    node_postprocessors=[reranker],
    llm=llm
)

# 3. Query
response = query_engine.query(query)
print(f"Query: {query}\n")
print("--- --------------- ---")
print(f"Response: {response}\n")

In [ ]:
print(f"Correct Answer: {answers[idx]}")